In [5]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

#idea change distance from fiber (lens)

In [6]:
MFD = (6.4 + 5.3) / 2 * 1e-6  #from fiber
w0 = MFD / 2                 # Strahltaille an der Faser

def get_abcd_matrix(l_m, f_m):
    space = np.array([[1.0,  l_m],
                      [0.0,  1.0]])
    lens  = np.array([[ 1.0,  0.0],
                      [-1.0/f_m, 1.0]])
    return lens @ space

def abcd_and_beam_interact(l_muem, wavelength_nm, f_mm, z_km):

    l = l_muem * 1e-6
    f = f_mm * 1e-3
    wavelength = wavelength_nm * 1e-9
    
    ABCD = get_abcd_matrix(l, f)
    A, B, C, D = ABCD[0,0], ABCD[0,1], ABCD[1,0], ABCD[1,1]
    
    print(f"=== SYSTEM-STATUS (Faser -> Linse) ===")
    print(f"Abstand Faser-Linse (l): {l_muem:.1f} μm")
    print(f"Brennweite Linse    (f): {f_mm:.1f} mm")
    print(f"Wellenlänge         (λ): {wavelength_nm:.1f} nm")
    print(f"Zielentfernung LEO  (z): {z_km:.1f} km\n")
    
    print(f"ABCD Matrix:")
    print(f"  [  {A:.4f}      {B:.4f} m   ]")
    print(f"  [  {C:.2f} 1/m  {D:.4f}     ]\n")
    print("-" * 55)

    zR = np.pi * w0**2 / wavelength
    
    w0_n = f * w0 / np.sqrt((f - l)**2 + zR**2)
    zR_n = np.pi * w0_n**2 / wavelength
    

    w_n = w0_n * np.sqrt(1 + (z_km * 1e3 / zR_n)**2)

    theta_n = wavelength / (np.pi * w0_n)
    
    print(f"Gauß-Strahl-Ergebnisse nach der Linse:")
    print(f"Neue Strahltaille (w0_n)     = {w0_n * 1e6:.2f} μm")
    print(f"Halbe Fernfeld-Divergenz (θ) = {theta_n * 1e6:.2f} μrad")
    print(f"Voll-Divergenz (2 * θ)       = {2 * theta_n * 1e6:.2f} μrad")
    print(f"Strahlradius am Satellit (w) = {w_n:.2f} m  (Fleckdurchmesser: {2*w_n:.2f} m)")
    print("-" * 55)


interact(
    abcd_and_beam_interact,

    l_muem=FloatSlider(
        value=36600.0, min=5*1e3, max=100*1e3, step=10,
        description="l (μm)"
    ),
    wavelength_nm=FloatSlider(
        value=1550, min=400.0, max=2000.0, step=1,
        description="λ (nm)"
    ),
    f_mm=FloatSlider(
        value=36.6, min=10.0, max=100.0, step=0.1,
        description="f (mm)"
    ),
    z_km=FloatSlider(
        value=500, min=0, max=600, step=1,
        description="z OGS-Sat (km)"
    )
)


interactive(children=(FloatSlider(value=36600.0, description='l (μm)', max=100000.0, min=5000.0, step=10.0), F…

<function __main__.abcd_and_beam_interact(l_muem, wavelength_nm, f_mm, z_km)>

In [7]:
#gib benötigtes l aus 

MFD = (6.4 + 5.3) / 2 * 1e-6  # info aus faser
w0 = MFD / 2                 # apparently

def get_abcd_matrix(l_m, f_m):
    space = np.array([[1.0,  l_m],
                      [0.0,  1.0]])
    lens  = np.array([[ 1.0,  0.0],
                      [-1.0/f_m, 1.0]])
    return lens @ space

def inverse_beam_solver(theta_half_urad, wavelength_nm, f_mm, z_km):
    # Umrechnung in SI-Einheiten
    theta_half = theta_half_urad * 1e-6
    wavelength = wavelength_nm * 1e-9
    f = f_mm * 1e-3
    z_m = z_km * 1e3
    
    zR = np.pi * w0**2 / wavelength

    w0_n_target = (2 * wavelength) / (np.pi * theta_half)
    
    wurzel_term = (f * w0 / w0_n_target)**2 - zR**2
    

    print(f"wished half-divergence: {theta_half_urad:.1f} μrad")
    print(f"wavelength λ:            {wavelength_nm:.1f} nm")
    print(f"focal length f:       {f_mm:.1f} mm\n")
    
    if wurzel_term < 0:
        print("not possible, chose bigger f or smaller divergence")
        print("-" * 55)
        return

    wurz_val = np.sqrt(wurzel_term)
    l1 = f - wurz_val
    
    ABCD = get_abcd_matrix(l1, f)
    
    zR_n = np.pi * w0_n_target**2 / wavelength
    w_sat = w0_n_target * np.sqrt(1 + (z_m / zR_n)**2)
    
    print(f"poition of fiber:  l = {l1 * 1e6:.1f} μm)")
    print("-" * 55)
    print(f"Zugehörige ABCD-Matrix für Lösung 1:")
    print(f"  [  {ABCD[0,0]:.4f}      {ABCD[0,1]:.4f} m   ]")
    print(f"  [  {ABCD[1,0]:.2f} 1/m  {ABCD[1,1]:.4f}     ]\n")
    print("-" * 55)
    print(f"Kontroll-Werte am Satellit (z = {z_km:.1f} km):")
    print(f"Erreichte Strahltaille (w0_n) = {w0_n_target * 1e6:.2f} μm")
    #print(f"Fleckdurchmesser am Satellit  = {2 * w_sat:.2f} m")
    print("-" * 55)

interact(
    inverse_beam_solver,
   
    theta_half_urad=FloatSlider(
        value=23, min=1, max=2000, step=1,
        description="Target θ/2 (μrad)"
    ),
    wavelength_nm=FloatSlider(
        value=1550, min=400.0, max=2000.0, step=1,
        description="λ (nm)"
    ),
    f_mm=FloatSlider(
        value=36.6, min=10.0, max=100.0, step=0.1,
        description="f (mm)"
    ),
    z_km=FloatSlider(
        value=500, min=300, max=600, step=1,
        description="z (km)"
    )
)

interactive(children=(FloatSlider(value=23.0, description='Target θ/2 (μrad)', max=2000.0, min=1.0, step=1.0),…

<function __main__.inverse_beam_solver(theta_half_urad, wavelength_nm, f_mm, z_km)>

In [8]:
#bei verschieben der linse: z_neu=z-l 
#in realität kein untershcied

In [9]:
import numpy as np
from ipywidgets import interact, FloatSlider

MFD = (6.4 + 5.3) / 2 * 1e-6  # info aus faser
w0 = MFD / 2                 

def get_abcd_matrix(l_m, f_m):
    space = np.array([[1.0,  l_m],
                      [0.0,  1.0]])
    lens  = np.array([[ 1.0,  0.0],
                      [-1.0/f_m, 1.0]])
    return lens @ space

def inverse_beam_solver(theta_half_urad, wavelength_nm, f_mm, z_km):
    # Umrechnung in SI-Einheiten
    theta_half = theta_half_urad * 1e-6
    wavelength = wavelength_nm * 1e-9
    f = f_mm * 1e-3
    z_m = z_km * 1e3
    
    zR = np.pi * w0**2 / wavelength

    # KORREKTUR: Für den Halbwinkel (theta_half) gehört hier KEINE 2 in den Zähler!
    w0_n_target = wavelength / (np.pi * theta_half)
    
    wurzel_term = (f * w0 / w0_n_target)**2 - zR**2
    
    print(f"=== INVERSE BERECHNUNG (KORRIGIERT) ===")
    print(f"Gewünschte Halbdivergenz θ: {theta_half_urad:.1f} μrad")
    print(f"Wellenlänge λ:              {wavelength_nm:.1f} nm")
    print(f"Brennweite f:               {f_mm:.1f} mm\n")
    
    if wurzel_term < 0:
        print("❌ Nicht möglich! Wähle größeres f oder kleinere Divergenz.")
        print("-" * 55)
        return

    wurz_val = np.sqrt(wurzel_term)
    l1 = f - wurz_val  # Position vor dem Fokus
    l2 = f + wurz_val  # Position hinter dem Fokus
    
    ABCD = get_abcd_matrix(l1, f)
    
    zR_n = np.pi * w0_n_target**2 / wavelength
    w_sat = w0_n_target * np.sqrt(1 + (z_m / zR_n)**2)
    
    # Berechne den echten mechanischen Stellweg relativ zur Brennweite (f - l)
    delta_l_muem = (f - l1) * 1e6
    
    print(f"💡 OPTOMECHANISCHE POSITIONIERUNG:")
    print(f"Absoluter Faserabstand l:  {l1 * 1e6:.2f} μm")
    print(f"Benötigter Stellweg (f-l): {delta_l_muem:.2f} μm  (<-- Das ist dein x bzw. delta!)")
    print(f"Alternative Position l2:   {l2 * 1e6:.2f} μm")
    print("-" * 55)
    print(f"Kontroll-Werte am Satellit (z = {z_km:.1f} km):")
    print(f"Erreichte Strahltaille (w0_n) = {w0_n_target * 1e6:.2f} μm")
    print(f"Strahlradius am Satellit (w)  = {w_sat:.2f} m")
    print("-" * 55)

interact(
    inverse_beam_solver,
    theta_half_urad=FloatSlider(
        value=23, min=1, max=1000, step=1,
        description="Target θ (μrad)"
    ),
    wavelength_nm=FloatSlider(
        value=1550, min=400.0, max=2000.0, step=1,
        description="λ (nm)"
    ),
    f_mm=FloatSlider(
        value=36.6, min=10.0, max=100.0, step=0.1,
        description="f (mm)"
    ),
    z_km=FloatSlider(
        value=500, min=300, max=600, step=1,
        description="z (km)"
    )
)

interactive(children=(FloatSlider(value=23.0, description='Target θ (μrad)', max=1000.0, min=1.0, step=1.0), F…

<function __main__.inverse_beam_solver(theta_half_urad, wavelength_nm, f_mm, z_km)>